In [ ]:
import re

import pandas as pd

### Nuclear pore (GO:0005643)

In [ ]:
hgnc = pd.read_csv(
    "../../data/HGNC/hgnc_complete_set.txt",
    sep="\t",
    dtype=str,
    usecols=["symbol", "prev_symbol", "uniprot_ids"],
)

In [ ]:
def unique_lookup(column):
    pairs = hgnc[["symbol", column]].dropna()
    pairs = pairs.assign(key=pairs[column].str.split("|"))
    pairs = pairs.explode("key")
    n_symbol = pairs.groupby("key")["symbol"].transform("nunique")
    pairs = pairs[n_symbol == 1]
    return dict(zip(pairs["key"], pairs["symbol"]))


APPROVED = set(hgnc["symbol"])
PREVIOUS = unique_lookup("prev_symbol")
UNIPROT = unique_lookup("uniprot_ids")


def harmonise(name):
    if not isinstance(name, str):
        return None
    tokens = [x.strip() for x in name.split("|") if x.strip()]
    for token in tokens:
        # upper-cased open reading frames, C1ORF123 -> C1orf123
        token = re.sub(r"^(C\w+)ORF(\d+)$", r"\1orf\2", token)
        if token in APPROVED:
            return token
        for lookup in [PREVIOUS, UNIPROT]:
            if token in lookup:
                return lookup[token]
    return tokens[0] if tokens else None


def write_reference(name, genes, background):
    background = sorted(set(background))
    genes = sorted(set(genes) & set(background))
    for suffix, values in [
        ("genes", genes),
        ("background", background),
    ]:
        with open(f"../../result/reference/{name}.{suffix}", "w") as f:
            f.write("\n".join(values))
    print(f"{name}: {len(genes)} genes, {len(background)} background")

In [ ]:
# is_a and part_of links from each GO term to its children
child = {}
term = None
with open("../../data/GO/go-basic.obo") as handle:
    for line in handle:
        line = line.strip()
        if line.startswith("["):
            term = None
        elif line.startswith("id: GO:"):
            term = line[4:]
        elif term and line.startswith("is_a: "):
            child.setdefault(line[6:16], set()).add(term)
        elif term and line.startswith("relationship: part_of "):
            child.setdefault(line[22:32], set()).add(term)

NUCLEAR_PORE = "GO:0005643"
pore_term = {NUCLEAR_PORE}
stack = [NUCLEAR_PORE]
while stack:
    for term in child.get(stack.pop(), ()):
        if term not in pore_term:
            pore_term.add(term)
            stack.append(term)
print(f"{len(pore_term)} GO terms under {NUCLEAR_PORE}")

# protein cellular component annotations, without NOT qualifiers
cc = []
with open("../../data/GO/goa_human.gaf") as handle:
    for line in handle:
        if line.startswith("!"):
            continue
        field = line.rstrip("\n").split("\t")
        if (
            field[0] == "UniProtKB"
            and field[8] == "C"
            and "NOT" not in field[3]
        ):
            cc.append((field[2], field[4]))
cc = pd.DataFrame(cc, columns=["symbol", "go_id"])
cc["gene"] = cc["symbol"].map(
    {s: harmonise(s) for s in cc["symbol"].unique()}
)
cc = cc[cc["gene"].notna()]

positive = cc.loc[cc["go_id"].isin(pore_term), "gene"]
write_reference("Hela_nuclearPore", positive, cc["gene"])